# 3. Azure AI Search 인덱싱
청크를 임베딩 후 Azure AI Search에 업로드

In [27]:
import os
import json
import time
from dotenv import load_dotenv
from openai import OpenAI
from azure.storage.blob import BlobServiceClient
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchField,
    SearchFieldDataType,
    SimpleField,
    SearchableField,
    VectorSearch,
    HnswAlgorithmConfiguration,
    VectorSearchProfile,
)
from azure.core.credentials import AzureKeyCredential
import base64

load_dotenv()

STORAGE_CONNECTION_STRING = os.getenv('AZURE_STORAGE_CONNECTION_STRING')
CONTAINER_NAME = os.getenv('AZURE_STORAGE_CONTAINER_NAME')
SEARCH_ENDPOINT = os.getenv('AZURE_SEARCH_ENDPOINT')
SEARCH_API_KEY = os.getenv('AZURE_SEARCH_API_KEY')
SEARCH_INDEX_NAME = os.getenv('AZURE_SEARCH_INDEX_NAME')
OPENAI_ENDPOINT = os.getenv('AZURE_OPENAI_ENDPOINT')
OPENAI_API_KEY = os.getenv('AZURE_OPENAI_API_KEY')
EMBEDDING_DEPLOYMENT = os.getenv('AZURE_OPENAI_EMBEDDING_DEPLOYMENT')

TARGET_YEAR = 2023
TARGET_COMPANIES = ['삼성전자', 'SK하이닉스', '현대자동차', 'NAVER', '카카오']

# 인덱싱할 전략 선택: 'fixed' or 'section'
CHUNK_STRATEGY = 'section'

openai_client = OpenAI(base_url=OPENAI_ENDPOINT + '/openai/v1', api_key=OPENAI_API_KEY)

print('환경변수 로드 완료')

환경변수 로드 완료


## AI Search 인덱스 생성

In [28]:
def create_index():
    index_client = SearchIndexClient(
        endpoint=SEARCH_ENDPOINT,
        credential=AzureKeyCredential(SEARCH_API_KEY)
    )

    fields = [
        SimpleField(name='id', type=SearchFieldDataType.String, key=True),
        SimpleField(name='company', type=SearchFieldDataType.String, filterable=True),
        SimpleField(name='year', type=SearchFieldDataType.Int32, filterable=True),
        SimpleField(name='strategy', type=SearchFieldDataType.String, filterable=True),
        SimpleField(name='section', type=SearchFieldDataType.String, filterable=True),
        SimpleField(name='chunk_id', type=SearchFieldDataType.Int32),
        SearchableField(name='text', type=SearchFieldDataType.String, analyzer_name='ko.microsoft'),
        SearchField(
            name='embedding',
            type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
            searchable=True,
            vector_search_dimensions=1536,
            vector_search_profile_name='hnsw-profile',
        ),
    ]

    vector_search = VectorSearch(
        algorithms=[HnswAlgorithmConfiguration(name='hnsw-algo')],
        profiles=[VectorSearchProfile(name='hnsw-profile', algorithm_configuration_name='hnsw-algo')],
    )

    index = SearchIndex(name=SEARCH_INDEX_NAME, fields=fields, vector_search=vector_search)

    result = index_client.create_or_update_index(index)
    print(f'인덱스 생성/업데이트 완료: {result.name}')

create_index()

인덱스 생성/업데이트 완료: dart-index


## 임베딩 생성

In [29]:
def get_embedding(text: str) -> list[float]:
    response = openai_client.embeddings.create(
        input=text,
        model=EMBEDDING_DEPLOYMENT
    )
    return response.data[0].embedding

# 테스트
test_emb = get_embedding('삼성전자 반도체 사업')
print(f'임베딩 차원: {len(test_emb)}')

임베딩 차원: 1536


## Blob에서 청크 로드

In [30]:
def load_chunks_from_blob(company_name: str, strategy: str) -> list[dict]:
    blob_service = BlobServiceClient.from_connection_string(STORAGE_CONNECTION_STRING)
    blob_name = f'chunks/{company_name}_{TARGET_YEAR}_{strategy}.json'
    blob_client = blob_service.get_blob_client(container=CONTAINER_NAME, blob=blob_name)

    try:
        data = blob_client.download_blob().readall()
        return json.loads(data.decode('utf-8'))
    except Exception as e:
        print(f'  {company_name} 청크 로드 실패: {e}')
        return []

## 인덱싱 실행

In [31]:
def make_doc_id(company: str, year: int, strategy: str, chunk_id: int) -> str:
    raw = f"{company}_{year}_{strategy}_{chunk_id}"
    return base64.urlsafe_b64encode(raw.encode()).decode()

In [32]:
BATCH_SIZE = 50  # AI Search 배치 업로드 크기

search_client = SearchClient(
    endpoint=SEARCH_ENDPOINT,
    index_name=SEARCH_INDEX_NAME,
    credential=AzureKeyCredential(SEARCH_API_KEY)
)

for company in TARGET_COMPANIES:
    print(f'[{company}] 인덱싱 중...')
    chunks = load_chunks_from_blob(company, CHUNK_STRATEGY)
    if not chunks:
        continue

    documents = []
    for chunk in chunks:
        embedding = get_embedding(chunk['text'])
        doc = {
            'id': make_doc_id(company, TARGET_YEAR, CHUNK_STRATEGY, chunk['chunk_id']),
            'company': company,
            'year': TARGET_YEAR,
            'strategy': CHUNK_STRATEGY,
            'section': chunk.get('section', ''),
            'chunk_id': chunk['chunk_id'],
            'text': chunk['text'],
            'embedding': embedding,
        }
        documents.append(doc)
        time.sleep(0.1)  # 임베딩 API 속도 제한 대비

    # 배치 업로드
    for i in range(0, len(documents), BATCH_SIZE):
        batch = documents[i:i+BATCH_SIZE]
        search_client.upload_documents(batch)
        print(f'  배치 업로드: {i+1}~{i+len(batch)} / {len(documents)}')

    print(f'  완료: {len(documents)}개 문서\n')

print('=== 인덱싱 완료 ===')

[삼성전자] 인덱싱 중...


  배치 업로드: 1~50 / 475
  배치 업로드: 51~100 / 475
  배치 업로드: 101~150 / 475
  배치 업로드: 151~200 / 475
  배치 업로드: 201~250 / 475
  배치 업로드: 251~300 / 475
  배치 업로드: 301~350 / 475
  배치 업로드: 351~400 / 475
  배치 업로드: 401~450 / 475
  배치 업로드: 451~475 / 475
  완료: 475개 문서

[SK하이닉스] 인덱싱 중...
  배치 업로드: 1~50 / 377
  배치 업로드: 51~100 / 377
  배치 업로드: 101~150 / 377
  배치 업로드: 151~200 / 377
  배치 업로드: 201~250 / 377
  배치 업로드: 251~300 / 377
  배치 업로드: 301~350 / 377
  배치 업로드: 351~377 / 377
  완료: 377개 문서

[현대자동차] 인덱싱 중...
  배치 업로드: 1~50 / 474
  배치 업로드: 51~100 / 474
  배치 업로드: 101~150 / 474
  배치 업로드: 151~200 / 474
  배치 업로드: 201~250 / 474
  배치 업로드: 251~300 / 474
  배치 업로드: 301~350 / 474
  배치 업로드: 351~400 / 474
  배치 업로드: 401~450 / 474
  배치 업로드: 451~474 / 474
  완료: 474개 문서

[NAVER] 인덱싱 중...
  배치 업로드: 1~50 / 457
  배치 업로드: 51~100 / 457
  배치 업로드: 101~150 / 457
  배치 업로드: 151~200 / 457
  배치 업로드: 201~250 / 457
  배치 업로드: 251~300 / 457
  배치 업로드: 301~350 / 457
  배치 업로드: 351~400 / 457
  배치 업로드: 401~450 / 457
  배치 업로드: 451~457 / 457
  완료: 4